# MOD10C1 Consolidated Output Viewer

Visualize the consolidated MOD10C1 (Snow-Covered Area) dataset.

- **Day_CMG_Snow_Cover**: fractional snow cover (0–100%)
- **Snow_Spatial_QA**: spatial QA flags

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

# --- Configuration ---
NC_PATH = (
    Path.home()
    / "data/nhf-runs/2026-03-16_gfv11_v0.1.0/data/raw/mod10c1_v061/mod10c1_v061_2005_consolidated.nc"
)
TIMESTEP = 0  # 0-364 for daily 2005


In [ ]:
ds = xr.open_dataset(NC_PATH)
print(ds)
print(f"\nTime range: {ds.time.values[0]} to {ds.time.values[-1]}")
print(f"Time steps: {ds.time.size}")
print(f"Grid: {ds.lat.size} lat x {ds.lon.size} lon")


## Single-day snow cover map

In [ ]:
sca = ds["Day_CMG_Snow_Cover"].isel(time=TIMESTEP)
time_label = str(ds.time.values[TIMESTEP])[:10]

fig, ax = plt.subplots(figsize=(14, 6))
sca.plot(ax=ax, cmap="Blues", vmin=0, vmax=100, cbar_kwargs={"label": "Snow Cover (%)"})
ax.set_title(f"Day_CMG_Snow_Cover — {time_label}")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

print(f"Stats for {time_label}:")
print(f"  min:   {float(sca.min()):.1f}")
print(f"  max:   {float(sca.max()):.1f}")
print(f"  mean:  {float(sca.mean()):.1f}")
print(f"  NaN %: {float(sca.isnull().mean()) * 100:.1f}%")


## Snow Spatial QA

In [ ]:
from matplotlib.colors import BoundaryNorm, ListedColormap

qa = ds["Snow_Spatial_QA"].isel(time=TIMESTEP)

# Snow_Spatial_QA lookup (NSIDC MOD10C1 v061 User Guide, Table 1)
QA_LABELS = {
    0: "Best",
    1: "Good",
    2: "OK",
    3: "Poor",
    4: "Other",
    237: "Inland water",
    239: "Ocean",
    250: "Cloud obscured water",
    252: "Antarctica mask",
    253: "Not mapped",
    254: "No retrieval",
    255: "Fill",
}

# Assign a color to each QA code
QA_COLORS = {
    0: "#2ecc71",   # green
    1: "#82e0aa",   # light green
    2: "#f4d03f",   # yellow
    3: "#e67e22",   # orange
    4: "#e74c3c",   # red
    237: "#3498db",  # blue
    239: "#1a5276",  # dark blue
    250: "#aeb6bf",  # grey
    252: "#d5dbdb",  # light grey
    253: "#717d7e",  # medium grey
    254: "#f5b7b1",  # pink
    255: "#2c3e50",  # near black
}

# Build colormap from only the QA values present in this timestep
qa_vals = qa.values[~np.isnan(qa.values)]
unique_vals = sorted(np.unique(qa_vals).astype(int))

colors = [QA_COLORS.get(v, "#999999") for v in unique_vals]
cmap = ListedColormap(colors)
bounds = [v - 0.5 for v in unique_vals] + [unique_vals[-1] + 0.5]
norm = BoundaryNorm(bounds, cmap.N)

fig, ax = plt.subplots(figsize=(14, 6))
im = qa.plot(ax=ax, cmap=cmap, norm=norm, add_colorbar=False)
cbar = plt.colorbar(im, ax=ax, ticks=unique_vals, spacing="uniform")
cbar.ax.set_yticklabels([f"{v} — {QA_LABELS.get(v, '?')}" for v in unique_vals])
ax.set_title(f"Snow_Spatial_QA — {time_label}")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

print("QA value distribution:")
unique, counts = np.unique(qa_vals, return_counts=True)
for u, c in sorted(zip(unique, counts), key=lambda x: -x[1]):
    label = QA_LABELS.get(int(u), "Unknown")
    print(f"  {int(u):>3d} ({label:<21s}): {c / qa_vals.size * 100:5.1f}%  (n={c:,})")


## Seasonal comparison (winter vs summer)

In [ ]:
winter = ds["Day_CMG_Snow_Cover"].sel(time=slice("2005-01-01", "2005-02-28")).mean(dim="time")
summer = ds["Day_CMG_Snow_Cover"].sel(time=slice("2005-06-01", "2005-08-31")).mean(dim="time")

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

winter.plot(ax=axes[0], cmap="Blues", vmin=0, vmax=100, cbar_kwargs={"label": "%"})
axes[0].set_title("Mean Snow Cover — Jan–Feb 2005")
axes[0].set_aspect("equal")

summer.plot(ax=axes[1], cmap="Blues", vmin=0, vmax=100, cbar_kwargs={"label": "%"})
axes[1].set_title("Mean Snow Cover — Jun–Aug 2005")
axes[1].set_aspect("equal")

plt.tight_layout()
plt.show()


## Annual mean snow cover

In [ ]:
annual_mean = ds["Day_CMG_Snow_Cover"].mean(dim="time")

fig, ax = plt.subplots(figsize=(14, 6))
annual_mean.plot(ax=ax, cmap="Blues", vmin=0, vmax=100, cbar_kwargs={"label": "Snow Cover (%)"})
ax.set_title("Mean Day_CMG_Snow_Cover — 2005 annual")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()


## Daily time series (CONUS spatial mean)

In [ ]:
daily_mean = ds["Day_CMG_Snow_Cover"].mean(dim=["lat", "lon"])

fig, ax = plt.subplots(figsize=(14, 4))
daily_mean.plot(ax=ax, color="#2980b9", linewidth=0.8)
ax.set_ylabel("Mean Snow Cover (%)")
ax.set_title("CONUS-mean daily snow cover — 2005")
ax.set_ylim(0, None)
plt.tight_layout()
plt.show()


## Manifest provenance check

In [ ]:
import json

manifest_path = NC_PATH.parent.parent.parent.parent / "manifest.json"
manifest = json.loads(manifest_path.read_text())
entry = manifest["sources"]["mod10c1_v061"]

print(f"Source key:    {entry['source_key']}")
print(f"Period:        {entry['period']}")
print(f"Variables:     {entry['variables']}")
print(f"Files:         {len(entry['files'])}")
print(f"Consolidated:  {list(entry.get('consolidated_ncs', {}).keys())}")
print(f"Last consolidated: {entry.get('last_consolidated_utc', 'N/A')}")


In [ ]:
ds.close()
